In [ ]:
import importlib
import time
import cv2
import numpy as np
from tdmclient import ClientAsync, aw
import kalman
import vision as vis



importlib.reload(vis)     # re-read the .py files at each run: Python keeps imported modules in memory
importlib.reload(kalman)

################   General Constants   ################
CAMERA_INDEX = 1
FRAME_WIDTH, FRAME_HEIGHT = 1280, 720
MM_PER_PIXEL = 1.0              # image scale: Thymio width (110 mm) / its width in pixels
MARKERS_AHEAD_OF_AXLE_MM = 0.0  # middle of the green and blue markers, measured forward from the wheel axle
SPEED_TO_MM_S = 0.3482          # Thymio speed unit -> mm/s, measured on our robot
KEY_COMMANDS = {                # key code of cv2.waitKeyEx (Windows) -> motor targets (left, right) [Thymio units]
    2490368: (100, 100),        # up arrow: forward
    2621440: (-100, -100),      # down arrow: backward
    2424832: (-60, 60),         # left arrow: turn left on the spot
    2555904: (60, -60),         # right arrow: turn right on the spot
    32: (0, 0),                 # space: stop
}
ESCAPE_KEY = 27                 # quit


def vision_to_kalman(coordinates):
    """detect_thymio ([u, v] px, theta rad, v downwards) -> wheel axle pose [x mm, y mm, theta rad], y upwards."""
    (u, v), theta_image = coordinates
    theta = -theta_image
    x = u * MM_PER_PIXEL - MARKERS_AHEAD_OF_AXLE_MM * np.cos(theta)
    y = (FRAME_HEIGHT - v) * MM_PER_PIXEL - MARKERS_AHEAD_OF_AXLE_MM * np.sin(theta)
    return np.array([x, y, theta])


def kalman_to_vision(pose):
    """Kalman pose [x mm, y mm, theta rad] -> ([u, v] px, theta rad) in the image axes, for drawing."""
    x, y, theta = pose
    return [int(x / MM_PER_PIXEL), int(FRAME_HEIGHT - y / MM_PER_PIXEL)], -theta


################   Thymio: motors set by events, wheel speeds read   ################
client = ClientAsync()
node = aw(client.wait_for_node())
aw(node.lock())
aw(node.register_events([("speed", 2)]))
aw(node.compile("""
onevent speed
    motor.left.target = event.args[0]
    motor.right.target = event.args[1]
"""))
aw(node.run())
aw(node.wait_for_variables({"motor.left.speed", "motor.right.speed"}))


def set_motor_targets(left, right):
    """Motor targets in Thymio units (-500 to 500)."""
    aw(node.send_events({"speed": [left, right]}))


def read_wheel_speeds():
    """Latest measured wheel speeds (left, right) [mm/s]."""
    client.process_waiting_messages()  # receive the variables sent by the robot since the last call
    return node.v.motor.left.speed * SPEED_TO_MM_S, node.v.motor.right.speed * SPEED_TO_MM_S


################   Camera   ################
cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

################   Main Loop   ################
ekf = None  # created at the first camera measurement
try:
    previous_time = time.monotonic()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.monotonic()
        dt = now - previous_time
        previous_time = now

        coordinates = vis.detect_thymio(frame)  # ([u, v] px, theta rad), or None if the robot is not seen
        if ekf is None:
            if coordinates is not None:
                ekf = kalman.ExtendedKalmanFilter(vision_to_kalman(coordinates))
        else:
            left_speed, right_speed = read_wheel_speeds()
            ekf.predict(left_speed, right_speed, dt)       # odometry, every frame
            if coordinates is not None:
                ekf.update(vision_to_kalman(coordinates))  # camera, when the robot is seen

        frame = vis.draw_direction(frame, coordinates)    # camera measurement
        if ekf is not None:
            center, theta = kalman_to_vision(ekf.pose)    # Kalman estimate, in blue
            tip = (int(center[0] + 80 * np.cos(theta)), int(center[1] + 80 * np.sin(theta)))
            cv2.arrowedLine(frame, tuple(center), tip, (255, 0, 0), 3)
            x, y, heading = ekf.pose
            sigma = np.sqrt(np.diag(ekf.covariance))
            source = "camera + odometry" if coordinates is not None else "odometry only"
            cv2.putText(frame, f"{source}: x {x:.0f} mm  y {y:.0f} mm  theta {np.degrees(heading):.0f} deg  "
                               f"sigma {sigma[0]:.1f} mm {np.degrees(sigma[2]):.1f} deg",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        cv2.putText(frame, "arrows: drive   space: stop   Esc: quit", (10, frame.shape[0] - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        cv2.imshow("Thymio", frame)

        key = cv2.waitKeyEx(1)  # keys are read only when the "Thymio" window has the focus
        if key == ESCAPE_KEY:
            break
        if key in KEY_COMMANDS:
            set_motor_targets(*KEY_COMMANDS[key])
finally:
    set_motor_targets(0, 0)
    aw(node.unlock())
    cap.release()
    cv2.destroyAllWindows()

KeyboardInterrupt: 